# BP-LLM UltraFeedback Eval + LoRA (r=1) Fine-tuning

Evaluates BP-LLM (unary JJ) win rate on openbmb/UltraFeedback,
then fine-tunes the policy model for 1 epoch with LoRA (r=1)
and re-evaluates Win Rate + ECE@10.

**Runtime:** Requires A100 GPU (40GB+). Use Colab Pro+ or higher.

In [1]:
!pip install peft datasets transformers accelerate tqdm -q

In [ ]:
import math
import itertools
import random
import gc
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple, Iterable, Union

import torch
import torch.nn.functional as F
from torch.nn.functional import log_softmax
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
from tqdm import tqdm

# =========================
# Config
# =========================
HF_TOKEN = None   # ← paste your HF token here if using gated models

# Models: Instruct as policy, Base as reference
MODEL_NAME     = "meta-llama/Llama-3.2-3B-Instruct"
REF_MODEL_NAME = "meta-llama/Llama-3.2-3B"  # Use same for reference (frozen)

# ---- UltraFeedback dataset ----
DATASET         = "openbmb/UltraFeedback"
DATASET_CONFIG  = None
SPLIT           = "train[:80%]"

# Train/Test split
TRAIN_FRAC      = 0.8
SEED            = 42
MIN_GAP         = 1.0  # require at least this helpfulness score gap

# Input limits
MAX_INPUT_TOKENS = 1024
MAX_GEN_TOKENS   = 512
BATCH_SIZE       = 4

# Chat template toggle
USE_CHAT_TEMPLATE = True

# Length normalization: "mean" or "sum"
LENGTH_NORM = "mean"

# BP hyperparams (defaults)
BETA            = 1.0
DELTA           = 0.0
TAU             = 1.0
GAMMA           = 1.0
JJ_INNER_STEPS  = 2

# Grid tuning
DO_TUNE         = True
BETAS           = [0.5, 1.0, 2.0]
DELTAS          = ['bco', 0.0]
TAUS            = [0.5, 1.0, 2.0]
GAMMAS          = [0.5, 1.0, 1.5, 2.0]
JJ_STEPS_LIST   = [1, 2, 3, 0]        # 0 => adaptive

# LoRA training config
LORA_R           = 1
LORA_ALPHA       = 16
LORA_DROPOUT     = 0.05
LEARNING_RATE    = 1e-4
TRAIN_BATCH_SIZE = 4      # Training batch size
GRAD_ACCUM_STEPS = 2
NUM_EPOCHS       = 1
WARMUP_STEPS     = 100
MAX_GRAD_NORM    = 1.0
SCORE_MAX_GEN_TOKENS = 256

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if torch.cuda.is_available() else torch.float32
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print(f"Device: {DEVICE} | Dtype: {DTYPE}")

In [ ]:
# =========================
# Dataset wrapper (for DataLoader)
# =========================
class PreferenceDataset(Dataset):
    def __init__(self, records: List[Dict]):
        self.records = records
    def __len__(self):
        return len(self.records)
    def __getitem__(self, idx):
        return self.records[idx]

def collate_fn(batch):
    return batch

def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# =========================
# UltraFeedback adapter
# =========================
def _extract_text(c: Dict) -> Optional[str]:
    if not isinstance(c, dict):
        return None
    for k in ("text", "response", "output", "completion", "content"):
        v = c.get(k)
        if isinstance(v, str) and v.strip():
            return v
    res = c.get("result")
    if isinstance(res, dict):
        v = res.get("text") or res.get("response") or res.get("output")
        if isinstance(v, str) and v.strip():
            return v
    return None

def _to_float(x) -> Optional[float]:
    try:
        return float(x)
    except Exception:
        return None

def _extract_score(c: Dict) -> Optional[float]:
    """
    Extract a scalar score from completion annotations.
    Prefer helpfulness rating; fall back to any numeric fields.
    """
    if not isinstance(c, dict):
        return None
    ann = c.get("annotations")
    if isinstance(ann, dict):
        help_ = ann.get("helpfulness")
        if isinstance(help_, dict):
            r = help_.get("Rating") or help_.get("rating") or help_.get("score")
            r = _to_float(r)
            if r is not None:
                return r
        for key in ("overall", "quality", "correctness", "honesty", "safety"):
            sub = ann.get(key)
            if isinstance(sub, dict):
                r = sub.get("Rating") or sub.get("rating") or sub.get("score")
                r = _to_float(r)
                if r is not None:
                    return r
        # Any flat numeric
        for _, v in ann.items():
            fv = _to_float(v)
            if fv is not None:
                return fv
    for k in ("score", "rating", "rank"):
        fv = _to_float(c.get(k))
        if fv is not None:
            return fv
    return None

def adapt_ultrafeedback(record: Dict, min_gap: float = MIN_GAP) -> Optional[Dict]:
    """
    Build (prompt, chosen, rejected) by picking the highest-scored completion
    vs the lowest-scored completion, and drop near-ties (< min_gap).
    """
    prompt = record.get("instruction")
    if not isinstance(prompt, str) or not prompt.strip():
        return None

    comps = record.get("completions")
    if not isinstance(comps, list) or len(comps) < 2:
        return None

    pairs = []
    for c in comps:
        text = _extract_text(c)
        score = _extract_score(c)
        if isinstance(text, str) and text.strip() and score is not None:
            pairs.append((text, float(score)))

    if len(pairs) < 2:
        return None

    pairs.sort(key=lambda t: t[1])  # low ... high
    lo_txt, lo_s = pairs[0]
    hi_txt, hi_s = pairs[-1]
    if hi_s - lo_s < min_gap:
        return None

    return {"prompt": prompt, "chosen": hi_txt, "rejected": lo_txt}

In [ ]:
# =========================
# Prompt rendering
# =========================
def render_prompt_with_policy_template(
    policy_tokenizer,
    prompt_or_messages: Union[str, List[Dict[str,str]]]
) -> str:
    if USE_CHAT_TEMPLATE and hasattr(policy_tokenizer, "apply_chat_template"):
        try:
            if isinstance(prompt_or_messages, list):
                msgs = prompt_or_messages
            else:
                msgs = [{"role": "user", "content": str(prompt_or_messages).strip()}]
            return policy_tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True
            )
        except Exception:
            pass
    if isinstance(prompt_or_messages, list):
        p = "\n".join(f"{m.get('role','user').upper()}: {(m.get('content') or '').strip()}" for m in prompt_or_messages)
    else:
        p = str(prompt_or_messages).strip()
    return p if p.endswith("\n") else (p + "\n")

def render_prompt_list(policy_tokenizer, prompts: List[Union[str, List[Dict[str,str]]]]) -> List[str]:
    return [render_prompt_with_policy_template(policy_tokenizer, p) for p in prompts]

def concat_prompt_response_text(prompt_text: str, response: str) -> Tuple[str, str]:
    full_text = prompt_text + ("" if prompt_text.endswith("\n") else "\n") + str(response).strip()
    return full_text, prompt_text

In [ ]:
# =========================
# Model loaders
# =========================
def load_causal_lm(model_id: str, token: Optional[str], dtype=DTYPE):
    """
    Load model on CPU by default.
    """
    tok = AutoTokenizer.from_pretrained(model_id, token=token, use_fast=True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_id, token=token, torch_dtype=dtype)
    model.eval()
    return tok, model

def load_policy_with_lora(model_id: str, token: Optional[str], dtype=DTYPE):
    """
    Load policy + LoRA adapter and move to GPU.
    """
    tok, base = load_causal_lm(model_id, token, dtype)
    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=["q_proj", "v_proj"]
    )
    policy = get_peft_model(base, lora_cfg)
    policy = policy.to(DEVICE)
    policy.print_trainable_parameters()
    return tok, policy

In [ ]:
# =========================
# Sequence scoring
# =========================

def sequence_logprob_stats_text(
    model,
    tokenizer,
    prompt_texts: List[str],
    responses: List[str],
    max_gen_tokens: int = MAX_GEN_TOKENS
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Returns (sum_logprobs, token_counts) for each (prompt, response) pair.
    If response was truncated, returns count=0 for that pair.
    NOTE: Returns tensors with gradients enabled for training.
    """
    sums_list = []  # Will hold tensors, not floats
    cnts_list = []  # Will hold integers

    for i in range(0, len(prompt_texts), BATCH_SIZE):
        batch_prompts = prompt_texts[i:i+BATCH_SIZE]
        batch_resps   = responses[i:i+BATCH_SIZE]

        batch_inputs, batch_prompt_lens = [], []
        for p, r in zip(batch_prompts, batch_resps):
            full_text, _ = concat_prompt_response_text(p, r)
            toks = tokenizer(
                full_text,
                truncation=True,
                max_length=min(MAX_INPUT_TOKENS + max_gen_tokens, tokenizer.model_max_length),
                return_tensors="pt",
                add_special_tokens=True,
            )
            tok_prompt = tokenizer(
                p,
                truncation=True,
                max_length=MAX_INPUT_TOKENS,
                return_tensors="pt",
                add_special_tokens=True,
            )
            p_len = tok_prompt["input_ids"].shape[-1]
            batch_inputs.append(toks)
            batch_prompt_lens.append(p_len)

        pad_id = tokenizer.pad_token_id or 0
        input_ids = torch.nn.utils.rnn.pad_sequence(
            [bi["input_ids"].squeeze(0) for bi in batch_inputs],
            batch_first=True, padding_value=pad_id
        )
        attention_mask = torch.nn.utils.rnn.pad_sequence(
            [bi["attention_mask"].squeeze(0) for bi in batch_inputs],
            batch_first=True, padding_value=0
        )
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)

        logits   = model(input_ids=input_ids, attention_mask=attention_mask).logits
        logprobs = log_softmax(logits.to(torch.float32), dim=-1)

        for b in range(input_ids.size(0)):
            p_len = batch_prompt_lens[b]
            ids   = input_ids[b]
            masks = attention_mask[b]
            L     = int(masks.sum().item())  # This .item() is OK, just for length check
            if p_len >= L:
                sums_list.append(torch.tensor(0.0, device=DEVICE))  # Keep as tensor
                cnts_list.append(0)
                continue
            targ = ids[p_len:L]
            pred = logprobs[b, p_len-1:L-1]
            lp = pred.gather(-1, targ.unsqueeze(-1)).squeeze(-1)
            sums_list.append(lp.sum())  # Keep as tensor, don't convert to float!
            cnts_list.append(len(targ))

    # Stack tensors to preserve gradients
    sums_tensor = torch.stack(sums_list)
    cnts_tensor = torch.tensor(cnts_list, dtype=torch.int32, device=DEVICE)
    return sums_tensor, cnts_tensor



@torch.no_grad()
def sequence_logprob_list(
    model,
    tokenizer,
    prompts: List[str],
    responses: List[str],
    length_norm: str = LENGTH_NORM,
) -> List[float]:
    """
    Returns mean/sum log-prob for each (prompt, response). If response was truncated
    (i.e., prompt consumed the full sequence), returns 0.0 for that item.
    """
    prompt_texts = render_prompt_list(tokenizer, prompts)
    sums, cnts = sequence_logprob_stats_text(model, tokenizer, prompt_texts, responses)
    vals = []
    for s, c in zip(sums.tolist(), cnts.tolist()):
        if c == 0:
            vals.append(0.0)
        else:
            vals.append(s / c if length_norm == "mean" else s)
    return vals

In [ ]:
# =========================
# BP-LLM posterior functions
# =========================
@dataclass
class BPParams:
    beta: float = BETA
    delta: float = DELTA
    tau: float = TAU
    gamma: float = GAMMA
    jj_steps: int = JJ_INNER_STEPS

def jj_lambda(xi: float) -> float:
    if xi < 1e-8:
        return 1.0 / 8.0
    return math.tanh(xi / 2.0) / (4.0 * xi)

def bp_unary_posterior(mu_prior: float, b: int, params: BPParams) -> Tuple[float, float]:
    tau2 = params.tau ** 2
    gamma_tilde = params.gamma * (2 * b - 1)
    mu_hat = mu_prior
    tau2_hat = tau2
    for _ in range(params.jj_steps):
        xi = abs(params.gamma) * math.sqrt(mu_hat * mu_hat + tau2_hat)
        lam = jj_lambda(xi)
        Lambda = (1.0 / tau2) + 2.0 * lam
        eta    = (mu_prior / tau2) + 0.5 * gamma_tilde
        mu_hat = eta / Lambda
        tau2_hat = 1.0 / Lambda
    return mu_hat, tau2_hat

def bp_unary_posterior_adaptive(mu_prior: float, b: int, params: BPParams,
                                tol: float = 1e-4, max_steps: int = 50) -> Tuple[float, float]:
    tau2 = params.tau ** 2
    gamma_tilde = params.gamma * (2 * b - 1)
    mu_hat = mu_prior
    tau2_hat = tau2
    steps = params.jj_steps if params.jj_steps and params.jj_steps > 0 else max_steps
    for _ in range(steps):
        mu_prev = mu_hat
        xi = abs(params.gamma) * math.sqrt(mu_hat * mu_hat + tau2_hat)
        lam = jj_lambda(xi)
        Lambda = (1.0 / tau2) + 2.0 * lam
        eta    = (mu_prior / tau2) + 0.5 * gamma_tilde
        mu_hat = eta / Lambda
        tau2_hat = 1.0 / Lambda
        if (params.jj_steps is None or params.jj_steps <= 0) and abs(mu_hat - mu_prev) < tol:
            break
    return mu_hat, tau2_hat

In [ ]:
# =========================
# ECE@10 calibration metrics
# =========================
def symmetric_ece_at_10(
    mu_ch_all: List[float],
    mu_rj_all: List[float],
    n_bins: int = 10
) -> Tuple[float, Dict]:
    """
    Compute symmetric ECE@10 from chosen/rejected posterior means.
    Returns (ece_value, details_dict).
    """
    import numpy as np

    mu_ch = np.array(mu_ch_all, dtype=np.float32)
    mu_rj = np.array(mu_rj_all, dtype=np.float32)

    # Compute confidence = sigmoid(mu_ch - mu_rj)
    logits = mu_ch - mu_rj
    confidence = 1.0 / (1.0 + np.exp(-logits))

    # Accuracy: 1 if mu_ch > mu_rj, else 0
    accuracy = (mu_ch > mu_rj).astype(np.float32)

    # Bin edges
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)

    details = []
    total_ece = 0.0
    N_total = len(confidence)

    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i+1]
        # Include right edge for last bin
        if i == n_bins - 1:
            mask = (confidence >= lo) & (confidence <= hi)
        else:
            mask = (confidence >= lo) & (confidence < hi)

        n_in_bin = int(mask.sum())
        if n_in_bin == 0:
            details.append({
                'bin': i+1,
                'range': f'[{lo:.1f}, {hi:.1f})',
                'count': 0,
                'avg_conf': 0.0,
                'avg_acc': 0.0,
                'ece_contrib': 0.0
            })
            continue

        avg_conf = float(confidence[mask].mean())
        avg_acc = float(accuracy[mask].mean())
        bin_ece = abs(avg_conf - avg_acc) * (n_in_bin / N_total)
        total_ece += bin_ece

        details.append({
            'bin': i+1,
            'range': f'[{lo:.1f}, {hi:.1f})',
            'count': n_in_bin,
            'avg_conf': avg_conf,
            'avg_acc': avg_acc,
            'ece_contrib': bin_ece
        })

    return total_ece, {'bins': details, 'n_total': N_total}

def print_ece_table(details: Dict, title: str = "ECE@10 Calibration"):
    print(f"\n{title}")
    print(f"{'='*70}")
    print(f"{'Bin':>3} {'Range':>12} {'Count':>6} {'Avg Conf':>9} {'Avg Acc':>9} {'ECE Contrib':>12}")
    print(f"{'-'*70}")
    for b in details['bins']:
        print(f"{b['bin']:3d} {b['range']:>12} {b['count']:6d} {b['avg_conf']:9.4f} {b['avg_acc']:9.4f} {b['ece_contrib']:12.6f}")
    print(f"{'='*70}")

In [ ]:
# =========================
# Evaluation & tuning
# =========================
def estimate_delta_bco(base_s: torch.Tensor, beta: float, N: int) -> float:
    r_pos = beta * base_s[:N]
    r_neg = beta * base_s[N:]
    return 0.5 * (float(r_pos.mean()) + float(r_neg.mean()))

@torch.no_grad()
def cache_pairwise_scores(
    records: List[Dict],
    model,
    ref_model,
    tokenizer
) -> Tuple[torch.Tensor, int]:
    """
    Returns base_s tensor shaped [2N] where first N are chosen scores, next N are rejected,
    and N is the number of *valid* pairs (both sides scored, count>0).
    Each score is (log pi - log pi_ref) with length normalization.
    """
    prompts_ch  = [r["prompt"] for r in records]
    prompts_rj  = [r["prompt"] for r in records]
    resps_ch    = [r["chosen"] for r in records]
    resps_rj    = [r["rejected"] for r in records]

    policy_ch = sequence_logprob_list(model, tokenizer, prompts_ch, resps_ch, LENGTH_NORM)
    policy_rj = sequence_logprob_list(model, tokenizer, prompts_rj, resps_rj, LENGTH_NORM)

    ref_ch = sequence_logprob_list(ref_model, tokenizer, prompts_ch, resps_ch, LENGTH_NORM)
    ref_rj = sequence_logprob_list(ref_model, tokenizer, prompts_rj, resps_rj, LENGTH_NORM)

    chosen_vals, rejected_vals = [], []
    for sc, sr, rc, rr in zip(policy_ch, policy_rj, ref_ch, ref_rj):
        if sc == 0.0 or sr == 0.0:  # truncated
            continue
        chosen_vals.append(sc - rc)
        rejected_vals.append(sr - rr)

    if len(chosen_vals) == 0:
        raise RuntimeError("No valid pairs after scoring. Consider increasing MAX_*_TOKENS or using 'sum' norm.")

    base_s = torch.tensor(chosen_vals + rejected_vals, dtype=torch.float32)
    N = len(chosen_vals)
    return base_s, N

def evaluate_with_cached(
    base_s: torch.Tensor, N: int, params: BPParams,
    use_adaptive_jj: bool = True, compute_ece: bool = False
) -> Dict:
    """
    Evaluate win rate and optionally ECE@10.
    """
    mu = (params.beta * base_s) - params.delta
    mu = mu.tolist()

    correct = 0
    mu_ch_all = []
    mu_rj_all = []


    for i in range(N):
            mu_ch = mu[i]          # Chosen prior
            mu_rj = mu[i + N]      # Rejected prior

            mu_ch_all.append(mu_ch)
            mu_rj_all.append(mu_rj)

        # Compare priors directly (no labels!)
            if mu_ch > mu_rj:
                correct += 1



    wr = 100.0 * correct / N
    result = {'win_rate': wr, 'n_pairs': N}

    if compute_ece:
        ece, ece_details = symmetric_ece_at_10(mu_ch_all, mu_rj_all, n_bins=10)
        result['ece10'] = ece
        result['ece_details'] = ece_details

    return result

def binom_ci_95(pct: float, N: int) -> Tuple[float, float]:
    p = pct / 100.0
    se = math.sqrt(p * (1 - p) / max(N, 1))
    lo = max(0.0, 100.0 * (p - 1.96 * se))
    hi = min(100.0, 100.0 * (p + 1.96 * se))
    return lo, hi

def grid_search(
    records: List[Dict], model, ref_model, tokenizer,
    betas: Iterable[float], deltas: Iterable[Optional[str]],
    taus: Iterable[float], gammas: Iterable[float],
    jj_steps_list: Iterable[int], use_adaptive_jj: bool = True,
    estimate_delta: bool = True, split_name: str = "TRAIN"
) -> Tuple[list, BPParams]:
    base_s, N = cache_pairwise_scores(records, model, ref_model, tokenizer)
    tried = []
    best = (-1.0, None)

    for beta, delta_spec, tau, gamma, jj_steps in itertools.product(
        betas, deltas, taus, gammas, jj_steps_list
    ):
        if estimate_delta and (delta_spec == 'bco'):
            delta = estimate_delta_bco(base_s, beta, N)
        else:
            delta = float(delta_spec)

        params = BPParams(beta=beta, delta=delta, tau=tau, gamma=gamma, jj_steps=jj_steps)
        result = evaluate_with_cached(base_s, N, params, use_adaptive_jj=use_adaptive_jj, compute_ece=False)
        wr = result['win_rate']
        lo, hi = binom_ci_95(wr, N)
        tried.append((wr, lo, hi, params))
        if wr > best[0]:
            best = (wr, params)

        print(f"[TUNE/{split_name}] WR={wr:.2f}% [{lo:.1f}, {hi:.1f}] "
              f"| beta={beta} delta={'bco' if delta_spec=='bco' else f'{delta:.4f}'} "
              f"| tau={tau} gamma={gamma} jj_steps={jj_steps}")

    best_wr, best_params = best
    lo, hi = binom_ci_95(best_wr, N)
    print(f"\n[BP-LLM Tuning ({split_name})] Best WR={best_wr:.2f}% [{lo:.1f}, {hi:.1f}] with "
          f"beta={best_params.beta} delta={best_params.delta:.4f} "
          f"tau={best_params.tau} gamma={best_params.gamma} jj_steps={best_params.jj_steps}")
    return tried, best_params

In [ ]:
def train_one_epoch(policy_model, ref_model, tokenizer, train_records, best_params):
    """
    Train policy for 1 epoch with LoRA using softplus margin loss on BP priors.
    Loss = softplus(-(mu_chosen - mu_rejected))
    where mu = beta * (log pi - log pi_ref) - delta
    """
    policy_model.train()

    # ─── PRE-CACHE all reference scores (ref is already on GPU here) ───
    print("Pre-caching reference model scores...")
    ref_model.eval()
    ref_cache = {}  # key: index → (ref_ch, ref_rj, valid)
    prompt_texts_all = [render_prompt_with_policy_template(tokenizer, r["prompt"]) for r in train_records]
    chosen_all   = [r["chosen"]   for r in train_records]
    rejected_all = [r["rejected"] for r in train_records]

    #with torch.no_grad():
     #   ref_ch_all = sequence_logprob_list(ref_model, tokenizer, [r["prompt"] for r in train_records], chosen_all, LENGTH_NORM)
      #  ref_rj_all = sequence_logprob_list(ref_model, tokenizer, [r["prompt"] for r in train_records], rejected_all, LENGTH_NORM)


    with torch.no_grad():
        ref_sum_ch, ref_cnt_ch = sequence_logprob_stats_text(ref_model, tokenizer, prompt_texts_all, chosen_all, SCORE_MAX_GEN_TOKENS)
        ref_sum_rj, ref_cnt_rj = sequence_logprob_stats_text(ref_model, tokenizer, prompt_texts_all, rejected_all, SCORE_MAX_GEN_TOKENS)

        # Convert to mean log-probs and cache as floats
        ref_ch_all = [(ref_sum_ch[i] / ref_cnt_ch[i]).item() if ref_cnt_ch[i] > 0 else 0.0 for i in range(len(train_records))]
        ref_rj_all = [(ref_sum_rj[i] / ref_cnt_rj[i]).item() if ref_cnt_rj[i] > 0 else 0.0 for i in range(len(train_records))]

    for i in range(len(train_records)):
        ref_cache[i] = (ref_ch_all[i], ref_rj_all[i])

    del ref_ch_all, ref_rj_all
    print(f"Cached {len(ref_cache)} reference scores. Offloading ref model to CPU...")

    # ─── Offload ref to CPU — no longer needed on GPU ───
    ref_model.cpu()
    clear_memory()

    # ─── Training setup ───
    # We need a dataloader that preserves original indices for cache lookup
    indexed_records = list(enumerate(train_records))
    random.shuffle(indexed_records)

    optimizer = torch.optim.AdamW(policy_model.parameters(), lr=LEARNING_RATE)
    num_training_steps = len(train_records) // (TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=WARMUP_STEPS,
        num_training_steps=num_training_steps
    )

    # ─── Training loop (1 epoch) ───
    step = 0
    optimizer.zero_grad()

    pbar = tqdm(range(0, len(indexed_records), TRAIN_BATCH_SIZE), desc="Training")
    for batch_start in pbar:
        batch_indexed = indexed_records[batch_start : batch_start + TRAIN_BATCH_SIZE]
        batch_indices = [idx for idx, _ in batch_indexed]
        batch_recs = [rec for _, rec in batch_indexed]



# ─── Forward pass: compute policy scores (with gradients) ───
        prompts = [r["prompt"] for r in batch_recs]
        prompt_texts = render_prompt_list(tokenizer, prompts)
        chosen  = [r["chosen"]  for r in batch_recs]
        rejected= [r["rejected"]for r in batch_recs]

        # Get policy scores as tensors (with gradients)
        policy_sum_ch, policy_cnt_ch = sequence_logprob_stats_text(policy_model, tokenizer, prompt_texts, chosen, SCORE_MAX_GEN_TOKENS)
        policy_sum_rj, policy_cnt_rj = sequence_logprob_stats_text(policy_model, tokenizer, prompt_texts, rejected, SCORE_MAX_GEN_TOKENS)

        # ─── Compute losses using cached ref scores ───
        losses = []
        for b_idx in range(len(batch_recs)):
            orig_idx = batch_indices[b_idx]
            ref_ch, ref_rj = ref_cache[orig_idx]

            # Skip if either side was truncated
            if policy_cnt_ch[b_idx] == 0 or policy_cnt_rj[b_idx] == 0 or ref_ch == 0.0 or ref_rj == 0.0:
                continue

            # Compute policy scores (mean log-prob)
            pc = policy_sum_ch[b_idx] / policy_cnt_ch[b_idx]
            pr = policy_sum_rj[b_idx] / policy_cnt_rj[b_idx]

            # BP prior: mu = beta * (log pi - log pi_ref) - delta
            mu_ch = best_params.beta * (pc - ref_ch) - best_params.delta
            mu_rj = best_params.beta * (pr - ref_rj) - best_params.delta

            # Margin loss: softplus(-(mu_ch - mu_rj))
            margin = mu_ch - mu_rj
            loss = F.softplus(-margin)
            losses.append(loss)


        if not losses:
            continue

        batch_loss = sum(losses) / len(losses)
        batch_loss = batch_loss / GRAD_ACCUM_STEPS
        batch_loss.backward()

        # ─── Gradient accumulation ───
        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(policy_model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        step += 1
        pbar.set_postfix({"loss": f"{batch_loss.item()*GRAD_ACCUM_STEPS:.4f}"})

    print("\nTraining complete.")
    return policy_model

In [ ]:
# =========================
# Main pipeline
# =========================
print("Loading policy model ...")
tok, policy = load_policy_with_lora(MODEL_NAME, HF_TOKEN)

print("\nLoading reference model ...")
_, ref = load_causal_lm(REF_MODEL_NAME, HF_TOKEN)

print(f"\nLoading dataset: {DATASET} [{SPLIT}] config={DATASET_CONFIG} ...")
if DATASET_CONFIG:
    ds = load_dataset(DATASET, name=DATASET_CONFIG, split=SPLIT)
else:
    ds = load_dataset(DATASET, split=SPLIT)

print("Columns:", ds.column_names)
if len(ds) > 0:
    sample0 = ds[0]
    print("Row[0] keys:", list(sample0.keys()))

# Adapt to (prompt, chosen, rejected) with top-vs-bottom + gap
adapted: List[Dict] = []
for rec in ds:
    a = adapt_ultrafeedback(rec, min_gap=MIN_GAP)
    if a and a["prompt"] and a["chosen"] and a["rejected"]:
        adapted.append(a)

print(f"Adapted examples (after gap>={MIN_GAP}): {len(adapted)}")
if not adapted:
    raise RuntimeError(
        "No valid examples after adapting UltraFeedback. "
        "Try reducing MIN_GAP, or inspect rows to tune _extract_*."
    )

# Deterministic split
g = torch.Generator().manual_seed(SEED)
idx = torch.randperm(len(adapted), generator=g).tolist()
split = int(len(idx) * TRAIN_FRAC)
train_idx, test_idx = idx[:split], idx[split:]
train_records = [adapted[i] for i in train_idx]
test_records  = [adapted[i] for i in test_idx]
print(f"Train records: {len(train_records)} | Test records: {len(test_records)}")

In [ ]:
# ---------- Tuning on TRAIN ----------
ref.cuda()
clear_memory()

if DO_TUNE:
    print("\nTuning on TRAIN...")
    _tries, best_params = grid_search(
        train_records, policy, ref, tok,
        betas=BETAS, deltas=DELTAS, taus=TAUS, gammas=GAMMAS,
        jj_steps_list=JJ_STEPS_LIST, use_adaptive_jj=True, estimate_delta=True, split_name="TRAIN"
    )
else:
    best_params = BPParams(beta=BETA, delta=DELTA, tau=TAU, gamma=GAMMA, jj_steps=JJ_INNER_STEPS)

ref.cpu()
clear_memory()

In [ ]:
# ---------- Pre-training TEST eval ----------
print(f"\n{'='*80}")
print("PRE-TRAINING TEST eval (with JJ labels) ...")
print(f"{'='*80}")

ref.cuda()
clear_memory()

base_s_pre, N_pre = cache_pairwise_scores(test_records, policy, ref, tok)
pre = evaluate_with_cached(base_s_pre, N_pre, best_params, use_adaptive_jj=True, compute_ece=True)

lo_pre, hi_pre = binom_ci_95(pre['win_rate'], N_pre)
print(f"[PRE-TRAIN] WR={pre['win_rate']:.2f}% [{lo_pre:.1f}, {hi_pre:.1f}] | ECE@10={pre['ece10']:.4f} | N={N_pre}")
print_ece_table(pre['ece_details'], title="ECE Calibration (TEST — pre-training)")

ref.cpu()
clear_memory()

In [ ]:
# ---------- LoRA Training (1 epoch) ----------
print(f"\n{'='*80}")
print("TRAINING ...")
print(f"{'='*80}")

ref.cuda()
clear_memory()

policy = train_one_epoch(policy, ref, tok, train_records, best_params)

ref.cpu()
clear_memory()

In [ ]:
# ---------- Post-training TEST eval ----------
print(f"\n{'='*80}")
print("POST-TRAINING TEST eval (with JJ labels) ...")
print(f"{'='*80}")

ref.cuda()
clear_memory()

base_s_post, N_post = cache_pairwise_scores(test_records, policy, ref, tok)
post = evaluate_with_cached(base_s_post, N_post, best_params, use_adaptive_jj=True, compute_ece=True)

lo_post, hi_post = binom_ci_95(post['win_rate'], N_post)
print(f"[POST-TRAIN] WR={post['win_rate']:.2f}% [{lo_post:.1f}, {hi_post:.1f}] | ECE@10={post['ece10']:.4f} | N={N_post}")
print_ece_table(post['ece_details'], title="ECE Calibration (TEST — post-training)")

ref.cpu()
clear_memory()

In [ ]:
# ---------- Summary ----------
print(f"\n{'='*80}")
print("SUMMARY (TEST set)")
print(f"{'='*80}")
print(f"  {'Metric':<20} {'Pre-train':>12} {'Post-train':>12} {'Delta':>12}")
print(f"  {'-'*20} {'-'*12} {'-'*12} {'-'*12}")

wr_pre = pre['win_rate']
wr_post = post['win_rate']
ece_pre = pre['ece10']
ece_post = post['ece10']

print(f"  {'Win Rate (%)':20} {wr_pre:12.2f} {wr_post:12.2f} {wr_post-wr_pre:+12.2f}")
print(f"  {'ECE@10':20} {ece_pre:12.4f} {ece_post:12.4f} {ece_post-ece_pre:+12.4f}")
print(f"  {'N pairs':20} {N_pre:12d} {N_post:12d}")
print(f"\n  Best params: beta={best_params.beta} delta={best_params.delta:.4f} tau={best_params.tau} gamma={best_params.gamma} jj_steps={best_params.jj_steps}")
print(f"{'='*80}")

In [ ]:
from google.colab import runtime
runtime.unassign()  # disconnects + releases the VM
